# F10 homework: from a bigram model to a tiny attention model

You will train two character-level language models on the opening of *Alice's Adventures in Wonderland*
(Lewis Carroll, 1865, public domain) and compare them:

- **Part A**: a bigram model built from **counts**, scored with cross-entropy and perplexity.
- **Part B**: the same bigram model as a **PyTorch network**, then a **one-block transformer** with causal
  self-attention, trained on the same text.
- **Part C**: compare losses and generated samples, and explain what you see.

**Where to run it.** Google Colab (File > Upload notebook) or local Jupyter with PyTorch installed. A GPU is
**not** needed: everything runs on the CPU in a few minutes. The text is included below, so nothing is
downloaded (except in the optional stretch task).

Each task has `TODO` cells and a check cell that prints `OK` when your code is right. Run the cells in order.

In [ ]:
import math
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(0)
print(torch.__version__)

## The text (given)

About 4,700 characters from the opening of Chapter 1. It is tiny by language-model standards (GPT-2 was
trained on about 40 GB of text), which is why everything trains in minutes, and why the samples will be
far from fluent.

In [ ]:
TEXT = "Alice was beginning to get very tired of sitting by her sister on the bank, and of having nothing to do: once or twice she had peeped into the book her sister was reading, but it had no pictures or conversations in it, \"and what is the use of a book,\" thought Alice \"without pictures or conversations?\"\n So she was considering in her own mind (as well as she could, for the hot day made her feel very sleepy and stupid), whether the pleasure of making a daisy-chain would be worth the trouble of getting up and picking the daisies, when suddenly a White Rabbit with pink eyes ran close by her.\n There was nothing so very remarkable in that; nor did Alice think it so very much out of the way to hear the Rabbit say to itself, \"Oh dear! Oh dear! I shall be late!\" (when she thought it over afterwards, it occurred to her that she ought to have wondered at this, but at the time it all seemed quite natural); but when the Rabbit actually took a watch out of its waistcoat-pocket, and looked at it, and then hurried on, Alice started to her feet, for it flashed across her mind that she had never before seen a rabbit with either a waistcoat-pocket, or a watch to take out of it, and burning with curiosity, she ran across the field after it, and fortunately was just in time to see it pop down a large rabbit-hole under the hedge.\n In another moment down went Alice after it, never once considering how in the world she was to get out again.\n The rabbit-hole went straight on like a tunnel for some way, and then dipped suddenly down, so suddenly that Alice had not a moment to think about stopping herself before she found herself falling down a very deep well.\n Either the well was very deep, or she fell very slowly, for she had plenty of time as she went down to look about her and to wonder what was going to happen next. First, she tried to look down and make out what she was coming to, but it was too dark to see anything; then she looked at the sides of the well, and noticed that they were filled with cupboards and book-shelves; here and there she saw maps and pictures hung upon pegs. She took down a jar from one of the shelves as she passed; it was labelled \"ORANGE MARMALADE\", but to her great disappointment it was empty: she did not like to drop the jar for fear of killing somebody underneath, so managed to put it into one of the cupboards as she fell past it.\n \"Well!\" thought Alice to herself, \"after such a fall as this, I shall think nothing of tumbling down stairs! How brave they'll all think me at home! Why, I wouldn't say anything about it, even if I fell off the top of the house!\" (Which was very likely true.)\n Down, down, down. Would the fall never come to an end? \"I wonder how many miles I've fallen by this time?\" she said aloud. \"I must be getting somewhere near the centre of the earth. Let me see: that would be four thousand miles down, I think--\" (for, you see, Alice had learnt several things of this sort in her lessons in the schoolroom, and though this was not a very good opportunity for showing off her knowledge, as there was no one to listen to her, still it was good practice to say it over) \"--yes, that's about the right distance--but then I wonder what Latitude or Longitude I've got to?\" (Alice had no idea what Latitude was, or Longitude either, but thought they were nice grand words to say.)\n Presently she began again. \"I wonder if I shall fall right through the earth! How funny it'll seem to come out among the people that walk with their heads downward! The Antipathies, I think--\" (she was rather glad there was no one listening, this time, as it didn't sound at all the right word) \"--but I shall have to ask them what the name of the country is, you know. Please, Ma'am, is this New Zealand or Australia?\" (and she tried to curtsey as she spoke--fancy curtseying as you're falling through the air! Do you think you could manage it?) \"And what an ignorant little girl she'll think me for asking! No, it'll never do to ask: perhaps I shall see it written up somewhere.\"\n Down, down, down. There was nothing else to do, so Alice soon began talking again. \"Dinah'll miss me very much to-night, I should think!\" (Dinah was the cat.) \"I hope they'll remember her saucer of milk at tea-time. Dinah my dear! I wish you were down here with me! There are no mice in the air, I'm afraid, but you might catch a bat, and that's very like a mouse, you know. But do cats eat bats, I wonder?\" And here Alice began to get rather sleepy, and went on saying to herself, in a dreamy sort of way, \"Do cats eat bats? Do cats eat bats?\" and sometimes, \"Do bats eat cats?\" for, you see, as she couldn't answer either question, it didn't much matter which way she put it."
print(len(TEXT))
print(TEXT[:300])

## Part A: a bigram model from counts

### Task A1: vocabulary, encode and decode

Build the character vocabulary as in the F10 tokens lesson:

- `chars`: the sorted list of distinct characters in `TEXT`, and `V = len(chars)`;
- `stoi` and `itos`: dictionaries from character to id and from id to character;
- `encode(s)` returns a list of ids; `decode(ids)` returns the string.

In [ ]:
# TODO
chars = None
V = None
stoi = None
itos = None

def encode(s):
    pass

def decode(ids):
    pass

In [ ]:
# Check A1
assert V == len(set(TEXT)) and chars == sorted(chars), "chars must be the sorted distinct characters"
assert decode(encode(TEXT)) == TEXT, "decode(encode(TEXT)) must give back the text"
assert encode("Alice") == [stoi[c] for c in "Alice"]
print("vocabulary size", V)
print("OK")

### The split (given)

The first 90% of the text is for training and the last 10% for validation. The validation text is short,
so validation numbers are a little noisy.

In [ ]:
data = torch.tensor(encode(TEXT), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]
print(len(train_data), len(val_data))

### Task A2: counts, probabilities and perplexity

1. `bigram_probs(ids, V, alpha)` returns a `(V, V)` float tensor `P` where `P[a, b]` is
   $(N[a, b] + \alpha) / \sum_c (N[a, c] + \alpha)$ and `N[a, b]` counts how often `b` follows `a` in `ids`.
2. `avg_nll(P, ids)` returns the mean of $-\ln P[x_{t-1}, x_t]$ over consecutive pairs, as a Python float.

Then fill `results`: for each `alpha` in `[0.01, 0.1, 1.0]`, store the train and validation losses
(**fit `P` on `train_data` only**), and pick the `alpha` with the lowest validation loss as `best_alpha`.

In [ ]:
def bigram_probs(ids, V, alpha):
    # TODO
    pass

def avg_nll(P, ids):
    # TODO
    pass

results = {}      # alpha -> (train_loss, val_loss)
best_alpha = None
# TODO: fill results and best_alpha

In [ ]:
# Check A2
_P = bigram_probs(torch.tensor([0, 1, 0, 2]), 3, 0.0)
assert torch.allclose(_P[0], torch.tensor([0.0, 0.5, 0.5])), "row 0 should be [0, 0.5, 0.5] for ids [0, 1, 0, 2] with alpha 0"
_P = bigram_probs(train_data, V, 0.1)
assert _P.shape == (V, V) and torch.allclose(_P.sum(dim=1), torch.ones(V)), "each row of P must sum to 1"
assert abs(avg_nll(torch.full((V, V), 1 / V), val_data) - math.log(V)) < 1e-5, "a uniform model must have loss ln V"
for a, (tr, va) in results.items():
    print(f"alpha {a:5}: train loss {tr:.3f}, val loss {va:.3f}, val perplexity {math.exp(va):.1f}")
assert set(results) == {0.01, 0.1, 1.0} and best_alpha in results
assert all(tr < va for tr, va in results.values()), "train loss should be below validation loss: fit P on train_data only"
assert best_alpha == min(results, key=lambda a: results[a][1])
P_counts = bigram_probs(train_data, V, best_alpha)
count_val_loss = avg_nll(P_counts, val_data)
assert count_val_loss < math.log(V) - 1, "the bigram model should beat the uniform model by a wide margin"
print("best alpha", best_alpha, "val loss", round(count_val_loss, 3), "uniform", round(math.log(V), 3))
print("OK")

### Task A3: generate from the count model

Write `sample_counts(P, start, n, g)`: start from the character `start`, then `n` times draw the next id
with `torch.multinomial(P[i], 1, generator=g)` and append its character. Return the string of length
`n + 1`.

In [ ]:
def sample_counts(P, start, n, g):
    # TODO
    pass

In [ ]:
# Check A3
_s = sample_counts(P_counts, "A", 200, torch.Generator().manual_seed(0))
assert isinstance(_s, str) and len(_s) == 201 and _s[0] == "A"
assert _s == sample_counts(P_counts, "A", 200, torch.Generator().manual_seed(0)), "the same seed must give the same text"
print(_s)
print("OK")

## Part B: neural models in PyTorch

### Task B1: batches

A training example is a window of `block_size` characters `x` and the same window shifted one to the
right, `y`: at every position the target is the next character. Write `get_batch(split, block_size,
batch_size, g)` that picks `batch_size` random start positions with
`torch.randint(len(d) - block_size, (batch_size,), generator=g)`, where `d` is `train_data` or
`val_data`, and returns `x` and `y`, each of shape `(batch_size, block_size)`.

In [ ]:
def get_batch(split, block_size, batch_size, g):
    d = train_data if split == "train" else val_data
    # TODO
    pass

In [ ]:
# Check B1
_x, _y = get_batch("train", 8, 4, torch.Generator().manual_seed(1))
assert _x.shape == (4, 8) and _y.shape == (4, 8)
assert torch.equal(_x[:, 1:], _y[:, :-1]), "y must be x shifted one position to the left"
_xv, _ = get_batch("val", 8, 4, torch.Generator().manual_seed(1))
assert decode(_xv[0]) in decode(val_data), "the val split must sample from val_data"
print(repr(decode(_x[0])), "->", repr(decode(_y[0])))
print("OK")

### The training loop (given)

`train` runs AdamW on random training batches and records the training loss; `estimate_loss` averages
the loss over many batches with the model in evaluation mode (dropout off). Every model below returns
`(logits, loss)` from `model(x, y)`.

In [ ]:
BLOCK, BATCH = 32, 32

@torch.no_grad()
def estimate_loss(model, split, iters=50):
    model.eval()
    g = torch.Generator().manual_seed(123)
    losses = [model(*get_batch(split, BLOCK, BATCH, g))[1].item() for _ in range(iters)]
    model.train()
    return sum(losses) / len(losses)

def train(model, steps, lr, seed=0):
    g = torch.Generator().manual_seed(seed)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.1)
    history, start = [], time.time()
    for step in range(steps + 1):
        x, y = get_batch("train", BLOCK, BATCH, g)
        _, loss = model(x, y)
        opt.zero_grad()
        loss.backward()
        opt.step()
        history.append(loss.item())
        if step % (steps // 5) == 0:
            print(f"step {step:5}: train {estimate_loss(model, 'train'):.3f}, "
                  f"val {estimate_loss(model, 'val'):.3f}  ({time.time() - start:.0f} s)")
    return history

### Task B2: the bigram model as a network

As in the F10 language modelling lesson: an `nn.Embedding(V, V)` whose
row for the current character *is* the vector of logits for the next one. `forward(idx, targets=None)`
takes `idx` of shape `(B, T)` and returns `logits` of shape `(B, T, V)` and the cross-entropy loss
(or `None` when `targets` is `None`). `F.cross_entropy` wants shapes `(N, V)` and `(N,)`, so reshape with
`.view(-1, V)` and `.view(-1)`.

In [ ]:
class BigramLM(nn.Module):
    def __init__(self, V):
        super().__init__()
        self.table = nn.Embedding(V, V)

    def forward(self, idx, targets=None):
        # TODO
        pass

torch.manual_seed(0)
bigram = BigramLM(V)
bigram_history = train(bigram, steps=1500, lr=0.05)

In [ ]:
# Check B2
_l, _loss = bigram(torch.zeros(2, 5, dtype=torch.long))
assert _l.shape == (2, 5, V) and _loss is None
bigram_val = estimate_loss(bigram, "val")
print(f"neural bigram val loss {bigram_val:.3f}, count bigram val loss {count_val_loss:.3f}")
assert abs(bigram_val - count_val_loss) < 0.15, "the trained network should match the count model closely"
print("OK")

### Task B3: a one-block transformer

Now let every position look back up to `BLOCK` characters with causal self-attention, exactly as in the
F10 attention and transformer lessons.

1. `Head.forward(x)`: `x` has shape `(B, T, d)`. Compute `q`, `k`, `v` with the three linear layers, the
   scores `q @ k.transpose(-2, -1) / sqrt(head_size)`, set future positions to `-inf` with
   `masked_fill(self.mask[:T, :T] == 0, float("-inf"))`, apply `F.softmax` on the last dimension, then
   `self.drop`, and return the weighted values, shape `(B, T, head_size)`.
2. `TinyGPT.forward(idx, targets=None)`:
   - `x = token embedding + position embedding` (positions `torch.arange(T)`);
   - attention sub-layer: `x = x + drop(proj(concat of every head applied to ln1(x)))`;
   - MLP sub-layer: `x = x + mlp(ln2(x))`;
   - `logits = lm_head(ln_f(x))`, and the loss as in `BigramLM`.

In [ ]:
class Head(nn.Module):
    def __init__(self, d, head_size, dropout):
        super().__init__()
        self.query = nn.Linear(d, head_size, bias=False)
        self.key = nn.Linear(d, head_size, bias=False)
        self.value = nn.Linear(d, head_size, bias=False)
        self.register_buffer("mask", torch.tril(torch.ones(BLOCK, BLOCK)))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        T = x.shape[1]
        # TODO
        pass


class TinyGPT(nn.Module):
    def __init__(self, V, d=32, n_heads=4, dropout=0.3):
        super().__init__()
        self.tok_emb = nn.Embedding(V, d)
        self.pos_emb = nn.Embedding(BLOCK, d)
        self.heads = nn.ModuleList([Head(d, d // n_heads, dropout) for _ in range(n_heads)])
        self.proj = nn.Linear(d, d)
        self.ln1, self.ln2, self.ln_f = nn.LayerNorm(d), nn.LayerNorm(d), nn.LayerNorm(d)
        self.mlp = nn.Sequential(nn.Linear(d, 4 * d), nn.ReLU(), nn.Linear(4 * d, d), nn.Dropout(dropout))
        self.lm_head = nn.Linear(d, V)
        self.drop = nn.Dropout(dropout)

    def forward(self, idx, targets=None):
        T = idx.shape[1]
        # TODO
        pass

In [ ]:
# Check B3 (before training): shapes and causality
torch.manual_seed(0)
_m = TinyGPT(V).eval()
_idx = torch.randint(V, (2, BLOCK))
_logits, _ = _m(_idx)
assert _logits.shape == (2, BLOCK, V)
_idx2 = _idx.clone()
_idx2[:, -1] = (_idx2[:, -1] + 1) % V                 # change only the last character
assert torch.allclose(_m(_idx2)[0][:, :-1], _logits[:, :-1], atol=1e-6), "earlier positions must not see later ones: check the causal mask"
assert not torch.allclose(_m(_idx2)[0][:, -1], _logits[:, -1]), "the last position should depend on its own input"
_short, _ = _m(_idx[:, :5])
assert _short.shape == (2, 5, V), "the model must also work on sequences shorter than BLOCK"
print("parameters:", sum(p.numel() for p in _m.parameters()))
print("OK")

Now train it (about half a minute on a CPU). Watch the gap between train and validation loss: with so
little text the model starts to memorise, which is why it has dropout.

In [ ]:
torch.manual_seed(0)
gpt = TinyGPT(V)
gpt_history = train(gpt, steps=1500, lr=2e-3)

In [ ]:
# Check B3 (after training)
gpt_val = estimate_loss(gpt, "val")
print(f"val loss: count bigram {count_val_loss:.3f}, neural bigram {bigram_val:.3f}, TinyGPT {gpt_val:.3f}")
assert gpt_val < bigram_val - 0.15, "the attention model should clearly beat the bigram model on validation loss"
plt.plot(bigram_history, label="bigram network", alpha=0.7)
plt.plot(gpt_history, label="TinyGPT", alpha=0.7)
plt.axhline(math.log(V), color="grey", ls="--", label="uniform, ln V")
plt.xlabel("training step")
plt.ylabel("training loss (nats per character)")
plt.legend()
plt.show()
print("OK")

### Task B4: generation with temperature

Write `generate(model, start, n, temperature, g)`: start from the ids of the string `start` as a tensor of
shape `(1, T)`. Then `n` times:

1. crop the context to the last `BLOCK` ids (the model has no position embeddings beyond that);
2. run the model and take the logits of the **last** position;
3. divide them by `temperature`, apply the softmax and draw one id with `torch.multinomial(..., 1,
   generator=g)`;
4. append it.

Return the decoded string (including `start`). Use `@torch.no_grad()` and put the model in `eval()` mode.

In [ ]:
@torch.no_grad()
def generate(model, start, n, temperature, g):
    model.eval()
    idx = torch.tensor([encode(start)], dtype=torch.long)
    # TODO
    pass

In [ ]:
# Check B4
_a = generate(gpt, "Alice ", 100, 1.0, torch.Generator().manual_seed(0))
assert isinstance(_a, str) and len(_a) == 106 and _a.startswith("Alice ")
assert _a == generate(gpt, "Alice ", 100, 1.0, torch.Generator().manual_seed(0)), "the same seed must give the same text"
_greedy1 = generate(gpt, "Alice ", 30, 1e-4, torch.Generator().manual_seed(1))
_greedy2 = generate(gpt, "Alice ", 30, 1e-4, torch.Generator().manual_seed(2))
assert _greedy1 == _greedy2, "at a tiny temperature, sampling should be greedy and ignore the seed"
for name, model in [("bigram", bigram), ("TinyGPT", gpt)]:
    for T in [0.5, 1.0]:
        print(f"--- {name}, temperature {T}")
        print(generate(model, "Alice ", 200, T, torch.Generator().manual_seed(0)))
print("OK")

## Part C: compare and explain

Answer in the markdown cell below, **using your own numbers and samples**:

1. Make a small table of validation loss and perplexity for the count bigram, the neural bigram and
   TinyGPT. Why do the two bigram models end up so close?
2. Compare the samples. Name two things TinyGPT gets right that the bigram model cannot, and explain them
   in terms of what each model can see.
3. How do the samples change between temperature 0.5 and 1.0? Relate this to the softmax.
4. TinyGPT's training loss keeps falling well below its validation loss. What is happening, and name two
   things that would help (think back to F9)?

*Your answers here.*

## Stretch (optional): pretrained word vectors

This downloads about 66 MB of GloVe word vectors (trained on Wikipedia and news text) with the `gensim`
library. On Colab, run the install cell first.

1. Print the 5 nearest neighbours of a few words of your choice with `wv.most_similar(word, topn=5)`.
2. Try analogies with `wv.most_similar(positive=["king", "woman"], negative=["man"], topn=5)`. Note that
   `most_similar` excludes the input words, as the F10 embeddings lesson
   warned. Find one analogy that works and one that fails.
3. Look for gendered associations: for several job words (for example nurse, engineer, doctor,
   programmer, teacher) compare `wv.similarity(job, "he")` with `wv.similarity(job, "she")`. Report what
   you find, including any results that surprise you, and connect it to
   module F11.

In [ ]:
# !pip install gensim
# import gensim.downloader as api
# wv = api.load("glove-wiki-gigaword-50")
# print(wv.most_similar("king", topn=5))